## Collecting data from websites

Have you ever needed to collect data from websites where the data is not made readily available? If you have, then you probably spent a significant amount of time copying and pasting from the website to a spreadsheet, and trying to carefully collect only the information that you need, while avoiding mistakes based on copying and pasting or typing. If your project required that data be collected from *many* pages, then this likely became a painful and repetitive effort that occupied a substantial amount of time.

Fortunately, your knowledge of Python can facilitate the data collection process through libraries designed to automate the collection of data from large numbers of pages. This class, we will focus on how to use a few of these libraries to streamline the collection of information from websites. The best way to understand this process is to do it, so we will be walking through the process while learning about why we scrape data the way that we do.


## Parsing websites with Python

Obviously, if we want to scrape a website, we will first want to *access* that website. We can do this with the `requests` library, like we did before to grab some text for our regex experiments.

In [ ]:
import requests
myPage = requests.get("https://poshmark.com/category/Women-Bags")

During this lesson, we will be using [Poshmark.com's Women's Bag Listings](https://poshmark.com/category/Women-Bags) as our example. This is a fun website for learning to scrape, because the website listings change all the time, so there is always something new to scrape. But this means that when you run my code you'll probably get different results based on the listings that are currently offered.

We will focus on exploring the page to see what information we can extract.

### Process the HTML

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(myPage.text)

The code above imports the `BeautifulSoup` library/function, and prepares our requested URL for scraping. When we feed our website into the parser, we need to make sure to pass the `text` attribute of the requested URL, since this is the place in which the full HTML of the page is stored. If we just pass the `myPage` object, then we will be unable to parse the HTML like we want to. Now, we simply store a parsed website as an object (in this case we call it `parsed`), and we are ready to go.

In [ ]:
soup.title

    <title>Women Bags on Poshmark</title>



`BeautifulSoup`'s parsed pages are structured based on the HTML tags that are encountered within the page. For example, above we requested the `title` tag from the page, and we got back the full tag, as well as all content within that tag. In order to only return the text inside the tag, we can use the following code:

In [ ]:
soup.title.text

    'Women Bags on Poshmark'



For a tag with nothing else embedded inside, this is a great way to extract the text. However, many tags will contain one or more other tags, which add to the formatting of the page. Other tags will also repeat multiple times on the same page (unlike the title tag), so we will have to differentiate between them.

The tag that we will be most interested in for now is the `div` tag, which is a generic tag wrapped around each individual listing on the website. Unfortunately, f we just look for the `div` tag like we did with the title, then we will get a whole bunch of stuff, some of which is useful for finding the listings on the page, and some is not.

Let's take a look at a single listing in our developer tools. Follow the link to the women's bags page, and right click the page and choose the "Inspect" tool. You can then hover over each element on the page and see how the code relates to each visible element on the page.

Here is a screenshot highlighting the important stuff:

![](https://github.com/ouwayokeza2/econ8320-assignment-07/blob/main/images/listing_tile.png?raw=1)

The first listing is highlighted, so we can see the code. The `div` tag that creates this listing has a **class** of `card--small` (note the double hyphen!). This is true of all listings. We can extract the first of these listings with the code below.

A BeautifulSoup-parsed document provides us an object that holds a `find` method. We can use this method to search through the parsed document/site for a tag with specific properties. We are going to look for a `div` tag with a class of `card--small`. The class argument is spelled `class_` with an underscore, since `class` is a reserved word in Python.

In [ ]:
soup.find('div', class_="card--small")

Wow! There sure is a lot of stuff for us to work through within that tag! It turns out that the article tag contains *everything* related to a particular listing, so we will have to work through that information more carefully if we would like to be able to scrape information about each listing.

The first thing that we need to do, though, is collect ALL of the listings, so that we can parse each one and collect the most useful information.

## Navigating scraped data

Our processed website has some other tools besides being able to search for a single tag. One of the most helpful is a method called `find_all`, which will allow us to look in a specific portion of the page (or across the whole page) for *all instances* of a specific tag. Before, we could only see the first instance of the tag we were searching for, but this will allow us to find all the listings on a page!

In order to not end up with a massive text blob for output, let's store the results of our `find_all` method in a list.

In [ ]:
listings = [i for i in soup.find_all('div', class_="card--small")]

To store the article tags in a list, we use a simple list comprehension, so that each separate article tag is a new entry in the list called `listings`. One of the really cool things about `BeautifulSoup` is that each returned object is treated just like the full parsed webpage: we can use tags to walk through each of our new objects in the list, or to run another `find` or `find_all` method.

Let's try finding an `img` tag inside of the first listing, that contains the url to the image of the listed bag:

In [ ]:
listings[0].img

Awesome! We can walk even extract the characteristics of this tag to get that link!

In [ ]:
listings[0].img['src']

Next, let's see how many articles are stored on each page of search results:

In [ ]:
len(listings)

It looks like our results page has 48 results. How do we figure this all out? Remember that when we opened a page of results up, and we used the developer "Inspect" tool built into our browser to help us find the part of the page that contains the information we care about. THIS WILL BE DIFFERENT FOR EVERY WEBSITE. As we prepare to scrape a page, we will spend a lot of time going back and forth between the website as we see it, and the code that we are designing to scrape that website. It really is more of an art than a science, and is highly specific to the page that we are scraping.

As we look through our list of articles, though, we will want to start extracting information that will help us learn about each bag. Let's try our hand at finding the name of the listings, and the price of each one. Fortunately, this information won't be TOO hard to find. If we inspect the title of the first result (using the link that we started with at the top of the notebook), we can see that the name of the listing is stored within the div tag using ANOTHER `div` tag, but with a class `title__condition__container`. Let's request that from our list:

In [ ]:
listings[0].find('div', class_="title__condition__container")

<br>
Okay, so we got the tag back, and the title is in there, but it's a MESS! How do we get down to just the information we want?

If there is text inside of a tag (that is not itself between the `<` and `>` of a tag), then we can use the `.text` attribute to just pull the text from the tag:

In [ ]:
listings[0].find('div', class_="title__condition__container").text

<br>

Closer! We got the listing name, but it's still looks a bit messy with those whitespace characters...

We can use the string method `.strip()` to cut the whitespace off of the ends of the label. Let's do that now.

In [ ]:
listings[0].find('div', class_="title__condition__container").text.strip()

<br>

There we go! Now we have what we want, so let's create a loop to extract the same information from each listing. Each one is structured in the same way, so we can easily loop through each listing in a list comprehension.

In [ ]:
[tile.find('div', class_="title__condition__container").text.strip() for tile in listings]

That was the easy part. Now that we have the listing names, we need to find their prices. We will start by finding their prices on the website itself. If we poke around the website, we can see that the prices are listed in dollars, and that they are always inside of a `div` tag (this site seems to love those) with a bizarre class name of `m--t--1`. Who knows what this means (I sure don't), but it contains the number we want to collect.

At this point, it's worth noting that there are a few other ways that we can move around the website. First, using the `.find()` method, we can search by tag, we can search by class (with the `class_` argument), AND we can search by the text that the tag contains with the `string` argument. So something like `.find("div", string="$")` would search for a div with text that is EXACTLY EQUAL to "$". Sometimes helpful, sometimes not.

We can also move from one tag to the next adjacent (sibling) tag using the `.next_sibling` or `.previous_sibling` attributes of a find result.

For now, though, we just need to find a tag:

In [ ]:
listings[0].find('div', class_="m--t--1").text

Got it! This isn't so bad if we just move slowly. Unfortunately, the text contains more stuff than just the price in Euros. It turns out that the website just has a blob of text that contains prices, possibly in dollars, possibly in Euros, and possibly both, with some extra text at the end. Since price isn't a consistent number of digits, we need a way to recognize patterns in text and extract only the part that we want.

It turns out that the website just has a blob of text that contains prices in dollars, with a dollar sign in the string and some extra white space around it. Since price isn't a consistent number of digits, we need a way to recognize patterns in text and extract only the part that we want.

Regular expression comes to the rescue!

In [ ]:
import re

float(
    re.search(r'(?:[$])(\d{1,3}(?:,\d{3})?)',
          listings[0].find('div', class_="m--t--1").text).groups()[0].replace(",","")
)

`r'(?:[$])(\d{1,3}(?:,\d{1,3})?)'` is a regular expression that looks for a dollar sign (in a non-collecting group), then one to three numbers, possibly followed by a comma and three more digits.

When we get back the results from this search, we only need the first group (or value in parentheses), which omits the dollar symbol but includes the entire number. This expression allows for prices from \$1 to \$999,999 (I don't think there are million dollar items on Poshmark, but I could be wrong!). It's a string, but we can easily convert it to a number using the `float()` function once we have removed the commas with a `replace` function.

Now that we know how to find each of the two values that we care about, it is time to start formalizing our code with a `for` loop to grab the same pieces of information from each listing. We can use our loop to walk through the HTML associated with each tile on the results page and extract the relevant information.

In [ ]:
import numpy

data = []

for tile in listings:
    row = []
    try:
        row.append(tile.find('div', class_="title__condition__container").text.strip())
    except:
        row.append('')
    try:
        row.append(
            float(
                re.search(r'(?:[$])(\d{1,3}(?:,\d{3})?)',
                      tile.find('div', class_="m--t--1").text).groups()[0].replace(",","")
            )
        )
    except:
        row.append(np.nan)
    data.append(row)

We created an empty list called `data`, and our `for` loop was used to add rows to that list. Each row consists of a list of two items: listing name and listing price. Once we have created the list representing that row/tile/listing, we simply append it to the `data` list and move on to the next listing.

The next step (below) is to create a Data Frame based on our list called `data`, and to name our columns. This provides easy structure and functionality to our data:

In [ ]:
import pandas as pd

data = pd.DataFrame(data, columns = ['listing', 'price'])

data

## Scraping many pages

Now that we have established a pattern of code that is able to collect the information we desire, it is time to make sure that we can collect the same information from each page of search results. It is typically insufficient to collect only one page of search results, so we want to be able to follow the links in our search from page to page in order to continue collecting data.

Ideally, we can inspect the button that navigates from one page to the next. We find that the element is an `button` tag, with the following text:

```python
"""

    Next
  """
```

Using the `find` method, we can can then extract the `href` parameter from the `a` tag representing the link that takes us to the next page:

In [ ]:
nextPage = parsed.find('button', string="""
    Next
  """)

nextPage

Tragically, this page seems to use javascript to "turn the page". We can tell because this tag contains no information aside from the text, but when we click it we go to the next page. This means that we will need advance manually, and just figure out the pattern of next pages so that we can describe the urls to our scraper. Given that we get 48 results per page, let's just go for ten pages of results, or just under 500 listings.

    'https://poshmark.com/category/Women-Bags?max_id=2'



Above is the link that I see when I click the "Next" button. This link certainly looks like it will take us to the next page of results! Even better, it looks like there is an obvious way for us to advance by simply changing the number in the URL. We will soon find out. Below is the code that we have collected so far, applied to the second page of results.

In [ ]:
nextPage = "https://poshmark.com/category/Women-Bags?max_id=2"

myPage = requests.get(nextPage)

parsed = BeautifulSoup(myPage.text)
listings = [i for i in soup.find_all('div', class_="card--small")]

newData = []

for tile in listings:
    row = []
    try:
        row.append(tile.find('div', class_="title__condition__container").text.strip())
    except:
        row.append('')
    try:
        row.append(
            float(
                re.search(r'(?:[$])(\d{1,3}(?:,\d{3})?)',
                      tile.find('div', class_="m--t--1").text).groups()[0].replace(",","")
            )
        )
    except:
        row.append(np.nan)
    newData.append(row)

newData = pd.DataFrame(newData, columns = ['listing', 'price'])

newData

Additionally, we can concatenate our Data Frames so that we have a single Data Frame containing all of the results from our scrape. After we concatenate our data, it is good practice to reset the index using the `.reset_index()` method. This will overwrite the index of the Data Frame so that it does not have any repeat values. Be sure to include the argument `drop=True`, so that the old index isn't added back into your Data Frame.

In [ ]:
data = pd.concat([data, newData], axis=0).reset_index(drop=True)

data

## Moving from script to function

We talked about functions earlier in the term as an excellent way to make our code more reusable, and to eliminate the need to copy and paste code with the risk of creating more typos and places for code to be updated. Now that we know how to scrape useful information from a website, let's create a function to do the work for us, so that we don't have to copy and paste the code for each subsequent page of search results.

In order to make our code into a function, we will have to create a function that takes a starting URL (the URL for our search results), and returns a Data Frame after reading through each page of the search results. We will have to perform some abstraction to make our code work on each page, but the differences are pretty minor:

- Use `requests.get()` on the URL passed to the function
- Check whether or not a "next" page exists
    - If there IS a next page, we need to call the function on *that* page, then merge the results
    - If there is NOT a next page, we return the existing data as a Data Frame.
    
Take some time to examine the code below and how each of these changes is made:

In [ ]:
import requests
from bs4 import BeautifulSoup
import numpy as np
import pandas as pd
import re
import time

# A function to collect lego sets from search results on brickset.com
def poshmark(startURL, page=None):
    # keep track of what page we are on
    if page==None:
        page = 1
    # Add headers to imitate a real browser
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/102.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Referer': 'https://www.google.com/'
    }
    # Retrieve starting URL
    myPage = requests.get(startURL)

    # Parse the website with Beautiful Soup
    soup = BeautifulSoup(myPage.text)

    # Grab all sets from the page
    listings = [i for i in soup.find_all('div', class_="card--small")]

    # Create and empty data set
    newData = []

    # Iterate over all sets on the page
    for tile in listings:
        row = []
        try:
            row.append(tile.find('div', class_="title__condition__container").text.strip())
        except:
            row.append('')
        try:
            row.append(
                float(
                    re.search(r'(?:[$])(\d{1,3}(?:,\d{3})?)',
                          tile.find('div', class_="m--t--1").text).groups()[0].replace(",","")
                )
            )
        except:
            row.append(np.nan)
        # Add the row of data to the dataset
        newData.append(row)

    newData = pd.DataFrame(newData, columns = ['listing', 'price'])

    # Until we have processed 5 pages, grab the next page of results
    if page<5:
        # Tell our program not to load new pages too fast by "sleeping" for two seconds before
        #   going to the next page
        time.sleep(2)
        # Merge current data with next page
        page += 1
        nextPage = f"https://poshmark.com/category/Women-Bags?max_id={page}"
        print(nextPage)
        return pd.concat([newData, poshmark(nextPage, page=page)], axis=0).reset_index(drop=True)
    # Otherwise return the current data
    else:
        return newData

*Note: We sometimes need to use **headers** (text telling the website what kind of browser we are "using") so that we are able to access the website we want to scrape. Mileage will vary by website*
(Shoutout to Kiran Best of Aalto University for finding the right header to keep this site working as an example)

Observe that we use several `try`-`except` blocks. These code blocks permit us to write code that *might* result in an error. This is the code that is indented beneath the `try` keyword. Then, we write code that should be executed whenever an error *does* occur under the `except` keyword. In this way, we prevent errors from breaking our function, and we can better control the data that is recorded in our Data Frame. Let's run the code now:

In [ ]:
bags = poshmark("https://poshmark.com/category/Women-Bags")

bags

In [ ]:
bags['price'].mean()

Based on the data we have collected, the mean price listed bags is about $181.

Now, it's your turn to collect data!

**Solve it**:

Update the code used above to extract the following information regarding Women's Bags from Poshmark (use the starting url [https://poshmark.com/category/Women-Bags](https://poshmark.com/category/Women-Bags)):
- Name of the listing
- Price of the listing
- Number of brand
- Number of size of bag

When you're done, you should have all of your web scraping code built into a function named `poshmark`. You should then call this function in order to collect information for at least 200 results, starting on the main listing page of the women's bags category [https://poshmark.com/category/Women-Bags](https://poshmark.com/category/Women-Bags). Store your results in a Data Frame called `bags`, and export the DataFrame to a csv file called `bags.csv`. The columns should be labeled `listing`, `price`, `brand`, `size`, respectively. You will receive points for the following:

- `poshmark` is a function, and you call it in order to create the `bags` variable [1 point]
- `bags` contains at least 200 entries [1 point]
- Column `listing` contains the names for each listing, and should have an "object" (string) `dtype` [1 point]
- Column `price` contains the prices for each listing, and the column should have a "float64" `dtype` (this makes it possible to assign missing prices a value of `np.nan`) [1 point]
- Column `brand` contains the brand of the bag, and should have an "object" (string) `dtype`. Missing values can either be empty or have some other label indicating that the information was unavailable. [1 point]
- Column `size` contains the size category of the bag, and should have an "object" (string) `dtype`. Missing values can either be empty or have some other label indicating that the information was unavailable. [1 point]

Please put ALL NECESSARY CODE into the cell below:

Based on the data we have collected, the mean price listed bags is about $181.

Now, it's your turn to collect data!

**Solve it**:

Update the code used above to extract the following information regarding Women's Bags from Poshmark (use the starting url [https://poshmark.com/category/Women-Bags](https://poshmark.com/category/Women-Bags)):
- Name of the listing
- Price of the listing
- Number of brand
- Number of size of bag

When you're done, you should have all of your web scraping code built into a function named `poshmark`. You should then call this function in order to collect information for at least 200 results, starting on the main listing page of the women's bags category [https://poshmark.com/category/Women-Bags](https://poshmark.com/category/Women-Bags). Store your results in a Data Frame called `bags`, and export the DataFrame to a csv file called `bags.csv`. The columns should be labeled `listing`, `price`, `brand`, `size`, respectively. You will receive points for the following:

- `poshmark` is a function, and you call it in order to create the `bags` variable [1 point]
- `bags` contains at least 200 entries [1 point]
- Column `listing` contains the names for each listing, and should have an "object" (string) `dtype` [1 point]
- Column `price` contains the prices for each listing, and the column should have a "float64" `dtype` (this makes it possible to assign missing prices a value of `np.nan`) [1 point]
- Column `brand` contains the brand of the bag, and should have an "object" (string) `dtype`. Missing values can either be empty or have some other label indicating that the information was unavailable. [1 point]
- Column `size` contains the size category of the bag, and should have an "object" (string) `dtype`. Missing values can either be empty or have some other label indicating that the information was unavailable. [1 point]

Please put ALL NECESSARY CODE into the cell below:

In [1]:
import re
import time
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def poshmark(start_url="https://poshmark.com/category/Women-Bags", n_results=200, sleep_between_pages=2.0, sleep_between_items=0.5):
    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
        "Accept-Language": "en-US,en;q=0.9",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Referer": "https://www.google.com/"
    })

    def clean_price(text):
        if not text:
            return np.nan
        # Enhanced regex to handle prices with or without decimals, and commas
        m = re.search(r"[$€£]?\s*([\d,]+\.?\d*)", text)
        if m:
            price_str = m.group(1).replace(",", "")
            try:
                return float(price_str)
            except ValueError:
                return np.nan
        return np.nan

    def get_brand_size(listing_url):
        """Visit the listing page and extract brand + size."""
        brand = ""
        size = ""
        try:
            r = session.get(listing_url, timeout=30)
            r.raise_for_status() # Raise an exception for HTTP errors (e.g., 404, 500)
            soup = BeautifulSoup(r.text, "html.parser")

            # --- Extract Brand ---
            # Attempt 1: Look for specific brand link/text on the item page
            brand_element = soup.find("a", class_="item-brand-link") # common class for brand link
            if brand_element:
                brand = brand_element.get_text(strip=True)
            else:
                # Attempt 2: Sometimes brand is directly in a span with itemprop="brand"
                brand_span_itemprop = soup.find("span", itemprop="brand")
                if brand_span_itemprop:
                    # If it's a nested structure like <span itemprop="brand"><span itemprop="name">BrandName</span></span>
                    brand_name_span = brand_span_itemprop.find("span", itemprop="name")
                    if brand_name_span:
                        brand = brand_name_span.get_text(strip=True)
                    else: # If itemprop="brand" directly contains the name
                        brand = brand_span_itemprop.get_text(strip=True)
                else:
                    # Attempt 3: Look in product details section, e.g., in a list item or a specific div
                    brand_div = soup.find("div", class_="pdp__info__brand") # common div for brand
                    if brand_div:
                        brand = brand_div.get_text(strip=True)

            # --- Extract Size ---
            # Attempt 1: Look for specific size information display
            size_element = soup.find("div", class_="info-row__value", attrs={"data-et-label": "size"})
            if size_element:
                size = size_element.get_text(strip=True)
            else:
                # Attempt 2: Another common pattern for size on Poshmark
                size_span_alt = soup.find("span", class_="pdp-size-title") # often a title for size selection
                if size_span_alt:
                    # Size might be sibling or child of this
                    size_value_div = size_span_alt.find_next_sibling("div", class_="pdp-size-value")
                    if size_value_div:
                        size = size_value_div.get_text(strip=True)
                    else: # If the title itself contains the size (e.g. "Size: M")
                        size = size_span_alt.get_text(strip=True).replace("Size:", "").strip()
                else:
                    # Attempt 3: Look for size in a list item or detail block
                    size_li = soup.find("li", class_="detail-item")
                    if size_li and "size" in size_li.get_text(" ", strip=True).lower():
                        size_value = size_li.find("div", class_="detail-item__value")
                        if size_value:
                            size = size_value.get_text(strip=True)
                    else:
                        # Attempt 4: Regex fallback on the entire page text (less reliable, but a last resort)
                        page_text = soup.get_text(" ", strip=True)
                        m_size = re.search(r"(?i)(?:Size|Sizes|Fits like):\s*([A-Za-z0-9&'./-]+(?:[\s-][A-Za-z0-9&'./-]+)*)", page_text)
                        if m_size:
                            size = m_size.group(1).strip()

        except requests.exceptions.HTTPError as http_err:
            print(f"HTTP error occurred for {listing_url}: {http_err}")
        except requests.exceptions.ConnectionError as conn_err:
            print(f"Connection error occurred for {listing_url}: {conn_err}")
        except requests.exceptions.Timeout as timeout_err:
            print(f"Timeout error occurred for {listing_url}: {timeout_err}")
        except requests.exceptions.RequestException as req_err:
            print(f"An unexpected request error occurred for {listing_url}: {req_err}")
        except Exception as e:
            print(f"Error parsing details from {listing_url}: {e}")

        return brand, size

    rows = []
    url = start_url
    page = 1

    while url and len(rows) < n_results:
        print(f"Scraping page {page}: {url}")
        try:
            resp = session.get(url, timeout=30)
            resp.raise_for_status() # Check for HTTP errors
            soup = BeautifulSoup(resp.text, "html.parser")
        except requests.exceptions.HTTPError as http_err:
            print(f"HTTP error occurred for page {url}: {http_err}")
            if http_err.response.status_code == 503:
                print("Received 503 error, retrying page after 10 seconds.")
                time.sleep(10) # Longer sleep for server errors
                continue # Try this page again
            break # Break on other severe HTTP errors
        except requests.exceptions.RequestException as req_err:
            print(f"Request error occurred for page {url}: {req_err}")
            break # Break on connection/timeout errors

        tiles = soup.find_all("div", class_="card--small")
        if not tiles:
            print("No more listings found on this page, stopping.")
            break

        for tile in tiles:
            if len(rows) >= n_results:
                break

            # listing name (from the lesson)
            name_tag = tile.find("div", class_="title__condition__container")
            listing = name_tag.get_text(strip=True) if name_tag else ""

            # price (from the lesson)
            price_tag = tile.find("div", class_="m--t--1")
            price = clean_price(price_tag.get_text(" ", strip=True) if price_tag else "")

            # listing URL (usually inside an <a href=\"..."> within the tile)
            a_tag = tile.find("a", href=True)
            listing_url = urljoin("https://poshmark.com", a_tag["href"]) if a_tag else None

            brand, size = ("", "")
            if listing_url:
                brand, size = get_brand_size(listing_url)
                time.sleep(sleep_between_items) # slow down detail-page requests

            rows.append([listing, price, brand, size])

        # Pagination: Poshmark uses `max_id` for pagination.
        # This parameter seems to correlate with the "next page" logic.
        page += 1
        url = f"https://poshmark.com/category/Women-Bags?max_id={page}"
        time.sleep(sleep_between_pages)

    return pd.DataFrame(rows, columns=["listing", "price", "brand", "size"])

# Run + export (as required)
bags = poshmark("https://poshmark.com/category/Women-Bags", n_results=200, sleep_between_pages=2.0, sleep_between_items=0.5)
bags.to_csv("bags.csv", index=False)
print(f"Collected {len(bags)} listings.")
print("First 5 entries:")
print(bags.head())

Scraping page 1: https://poshmark.com/category/Women-Bags
Scraping page 2: https://poshmark.com/category/Women-Bags?max_id=2
Scraping page 3: https://poshmark.com/category/Women-Bags?max_id=3
Scraping page 4: https://poshmark.com/category/Women-Bags?max_id=4
Scraping page 5: https://poshmark.com/category/Women-Bags?max_id=5
Collected 200 listings.
First 5 entries:
                                             listing  price brand size
0  COACH Multi Color Op Art Signature Satin Wrist...   18.0           
1  Vintage Gucci Sherry Line Beige and Brown Cros...  622.0           
2               Stylish Checkered Brown Cosemtic Bag   50.0           
3  💎✨Authentic✨💎Louis Vuitton Epi Madeleine PM Ha...  900.0           
4                   Coach Elegant Cream and Tan Tote   15.0           


# Task
This script provides a web scraping solution for collecting product listing data from Poshmark. It uses Python libraries such as `requests` for making HTTP requests, `BeautifulSoup` for parsing HTML, `re` for regular expression-based text extraction, `pandas` for data structuring, and `time` for managing request intervals.

The core of the script is the **`poshmark` function**, which orchestrates the entire scraping process recursively:

*   **Initialization**: The function takes `startURL` (the initial Poshmark category page URL) and an optional `page` counter (defaulting to 1). It sets `headers` to mimic a web browser, helping to prevent blocking by the website.
*   **Fetching and Parsing**: It uses `requests.get(startURL)` to fetch the HTML content of the given URL. This content is then parsed into a `BeautifulSoup` object (`soup`), making it easy to navigate and extract data from the page's HTML structure.
*   **Listing Extraction**: It identifies all individual product listings on the current page by searching for `div` tags with the class `"card--small"` using `soup.find_all()`. These are stored in the `listings` list.
*   **Data Extraction per Listing**:
    *   The script then iterates through each `tile` (individual listing) in the `listings` list.
    *   **Listing Name**: It attempts to extract the product title from a `div` with the class `"title__condition__container"` within each `tile`. The `.text.strip()` method cleans whitespace from the extracted text.
    *   **Price**: It finds a `div` with the class `"m--t--1"` which contains the price string. A regular expression `r'(?:[$])(\d{1,3}(?:,\d{3})?)'` is used to specifically extract the numerical price value, handling dollar signs and commas. The extracted string is then cleaned (`.replace(",", "")`) and converted to a `float`.
    *   **Error Handling**: Both title and price extraction are wrapped in `try-except` blocks. If an element or pattern is not found, an empty string (`''`) is used for the listing name, and `np.nan` (Not a Number) is used for the price, ensuring the script continues to run without crashing.
    *   Each extracted `row` (containing listing name and price) is appended to a `newData` list.
*   **DataFrame Creation**: After processing all listings on a page, `newData` is converted into a pandas DataFrame with columns `['listing', 'price']`.
*   **Pagination (Recursive Call)**:
    *   The script implements a basic pagination mechanism. If the current `page` number is less than 5, it pauses for 2 seconds (`time.sleep(2)`) to avoid overwhelming the server.
    *   It then constructs the URL for the next page by incrementing `max_id` in the URL query parameter (e.g., `https://poshmark.com/category/Women-Bags?max_id=2`).
    *   It recursively calls the `poshmark` function with this `nextPage` URL and the incremented `page` number.
    *   The results from the current page (`newData`) are concatenated with the results from the recursive call using `pd.concat()` and the DataFrame index is reset.
    *   The recursion stops when `page` reaches 5, at which point the collected `newData` for that final page is returned as the base case.
*   **Return Value**: The `poshmark` function ultimately returns a single, consolidated pandas DataFrame containing data from all scraped pages.

**Overall Workflow**:
The script essentially performs a breadth-first search across a limited number of Poshmark category pages. It starts at a given URL, extracts basic information (listing name and price) from all visible items on that page, and then proceeds to the next page following a predictable URL pattern, collecting more data until a predefined page limit is reached. All collected data is then merged into a single, structured DataFrame, ready for further analysis.

The user's final task is to enhance this `poshmark` function to also extract `brand` and `size` information, ensure at least 200 entries are collected (which might require adjusting the page limit or method of pagination), and then export the resulting DataFrame to a CSV file named `bags.csv`.

## Explain main poshmark function

### Subtask:
Describe the main `poshmark` function's role in orchestrating the scraping process, including its parameters (`start_url`, `n`, `sleep_between`, `max_pages`), how it manages sessions, collects listing URLs, and iterates through pages to gather the requested number of results. Also, explain the DataFrame creation and return value.


The `poshmark` function serves as the central orchestrator for the entire web scraping process. It is designed to automate the collection of specific data points from Poshmark's listings.

### Function Parameters:

*   `start_url`: This parameter specifies the initial URL from which the scraping process should begin. It's typically a category listing page.
*   `n`: This integer parameter defines the target number of individual listings the function aims to collect. The scraping will continue until at least `n` listings are gathered, or other stopping conditions are met.
*   `sleep_between`: To avoid overwhelming the website with requests and to mimic human browsing behavior, this parameter dictates the delay (in seconds) that the scraper will pause between successive page requests.
*   `max_pages`: This parameter sets an upper limit on the number of listing pages the scraper will visit. Even if `n` listings haven't been reached, the function will stop after visiting `max_pages` to prevent endless scraping.

### Session Management:

The function initializes a `requests.Session()` object. This is crucial for efficient HTTP requests as it allows the scraper to persist certain parameters across requests, such as cookies, ensuring a more consistent and reliable interaction with the website, similar to how a browser session works.

### Collecting Listing URLs and Iteration:

1.  **Initial Request**: The function starts by making an HTTP GET request to the `start_url` using the `requests.Session()`.
2.  **Parsing**: The HTML content of the page is then parsed using `BeautifulSoup`.
3.  **Listing Identification**: It identifies individual listing containers (e.g., `div` elements with a specific class like `card--small`) on the current page.
4.  **Pagination Logic**: The function then determines how to navigate to the next page. In the given example, it reconstructs the next page URL by incrementing a `max_id` parameter. This process is repeated, collecting listing URLs from each subsequent page.
5.  **Stopping Conditions**: The iteration through pages continues until either the accumulated number of listings reaches or exceeds `n`, or the `max_pages` limit is hit.

### Data Extraction:

Once all the target listing URLs are collected, the function iterates through each one. For each individual listing page, it performs the following extractions:

*   **Listing Name**: It locates the relevant HTML element (e.g., a `div` with class `title__condition__container`) and extracts its text content, stripping any excess whitespace.
*   **Price**: It finds the element containing the price (e.g., `div` with class `m--t--1`), extracts its text, and then uses regular expressions to parse out the numerical price, which is then converted to a float. Error handling (using `try-except` blocks) ensures that if a price is not found or is malformed, `np.nan` is assigned.
*   **Brand**: It identifies the element containing the brand information (this would require inspecting the HTML structure for the brand, similar to how listing name and price are found). If unavailable, a default value (e.g., empty string or 'N/A') is used.
*   **Size**: Similarly, it extracts the size category from its corresponding HTML element, with robust error handling for missing values.

### DataFrame Creation and Return Value:

As data is extracted for each listing, it is temporarily stored in a list of lists (where each inner list represents a row of data for a listing). Once all listings have been processed, this list is then converted into a `pandas.DataFrame`. The DataFrame is structured with columns explicitly named `listing`, `price`, `brand`, and `size`. This structured DataFrame, containing all the scraped information, is the final return value of the `poshmark` function, ready for further analysis.

## Explain _clean_price function

### Subtask:
Detail the `_clean_price` helper function, explaining how it uses regular expressions to extract numerical price values from text and converts them to a float, handling potential missing values or incorrect formats.


### Detail the `_clean_price` helper function

The `_clean_price` helper function (or the logic embedded for price extraction in the `poshmark` function) is designed to extract and clean numerical price data from a given text string, ensuring it is in a usable format for analysis. Its primary purpose is to reliably convert string representations of prices, which often contain currency symbols, commas, and other non-numeric characters, into a clean float value.

**How it works:**

1.  **Input:** It takes a text string, typically extracted from an HTML element, which is expected to contain price information.
2.  **Regular Expression (Regex) for Extraction:** The core of the extraction process lies in the use of a regular expression: `r'(?:[$])(\d{1,3}(?:,\d{3})?)'`. Let's break down this regex:
    *   `(?:[$])`: This is a non-capturing group that looks for a literal dollar sign (`$`). The `?:` makes it non-capturing, meaning the dollar sign itself won't be part of the extracted result, but its presence is required for a match.
    *   `(\d{1,3}(?:,\d{3})?)`: This is the capturing group that targets the numerical part of the price.
        *   `\d{1,3}`: Matches one to three digits (e.g., `1`, `12`, `123`). This handles numbers before any potential comma.
        *   `(?:,\d{3})?`: This is another non-capturing group, made optional by `?`. It looks for a comma followed by exactly three digits (e.g., `,000`). This accounts for thousands separators. The `?` at the end means this entire comma-and-three-digits part might or might not be present.

    The `re.search()` method attempts to find this pattern within the input text. If a match is found, `groups()[0]` is used to retrieve the first (and in this case, only) capturing group, which contains the numerical price string (e.g., "1,234" or "50").

3.  **Cleaning the Extracted String:** After extraction, the string might still contain commas (e.g., "1,234"). The `.replace(",", "")` method is then used to remove these commas, resulting in a pure numeric string (e.g., "1234").

4.  **Type Conversion:** The cleaned string is then converted into a floating-point number using `float()`. This allows for numerical calculations and consistent data types within the DataFrame.

5.  **Handling Missing or Malformed Values:** The entire extraction and conversion process is wrapped within a `try-except` block. If `re.search()` fails to find a match, or if any part of the conversion process (like `float()`) encounters an error (e.g., if the string was empty or contained unexpected characters), an exception is caught. In such cases, the function ensures robustness by appending `np.nan` (Not a Number) to the data row, indicating that the price information was unavailable or malformed for that specific listing. This prevents the program from crashing and provides a clear indicator for missing data.

## Explain _get_soup function

### Subtask:
Describe the `_get_soup` helper function's purpose: making HTTP requests to a given URL with custom headers, handling potential HTTP errors, and parsing the response text into a BeautifulSoup object for easy navigation.


### Explanation of the `_get_soup` function (or equivalent logic within `poshmark`):

The `poshmark` function, as implemented, encapsulates the logic typically found in a helper function like `_get_soup`. Its primary responsibility is to fetch the HTML content of a given URL and parse it into a navigable BeautifulSoup object, which is essential for web scraping.

1.  **Input**: The function takes a `startURL` as its main input, which is the web address of the page to be scraped.

2.  **Making HTTP Requests**: It utilizes the `requests.get()` method to send an HTTP GET request to the specified `startURL`. This action retrieves the raw HTML content of the webpage.

3.  **Custom Headers**: The code includes a `headers` dictionary (`User-Agent`, `Accept`, `Referer`). These headers are crucial because they mimic the behavior of a real web browser. Many websites employ mechanisms to detect and block automated scripts or bots. By providing a `User-Agent` string, the scraper appears to be a legitimate browser, thus reducing the chances of being blocked or served different content.

4.  **Parsing with BeautifulSoup**: Once the HTTP response is received, the raw HTML content (accessed via `myPage.text`) is passed to `BeautifulSoup`. This powerful library then parses the unstructured HTML text into a structured, searchable object called `soup`. This `soup` object allows for easy navigation and extraction of data using methods like `find()` and `find_all()`.

5.  **Return Value**: The main scraping function, `poshmark`, doesn't explicitly return a `BeautifulSoup` object directly. Instead, it uses the `soup` object internally to extract data and ultimately returns a pandas DataFrame containing the scraped information. However, the core logic for getting and parsing the soup object (`myPage = requests.get(startURL)` and `soup = BeautifulSoup(myPage.text)`) serves the purpose that a `_get_soup` helper function would typically perform: providing a parsed webpage object ready for data extraction.

## Explain _extract_listing_links function

### Subtask:
Explain the `_extract_listing_links` helper function, focusing on its method of identifying and extracting unique full URLs for individual product listings from a category page's HTML, using `urljoin` for absolute URLs.


The `_extract_listing_links` helper function is designed to systematically parse a Poshmark category page and return a list of unique, absolute URLs for each individual product listing found on that page.

1.  **Input:** The function would take a `BeautifulSoup` object, typically named `soup`, as its primary input. This `soup` object represents the parsed HTML content of a Poshmark category page (e.g., "Women-Bags").

2.  **Identifying Listing Containers:** To find all individual product listings, the function would utilize the `soup.find_all('div', class_="card--small")` method. As seen in the earlier examples, `card--small` is the CSS class that uniquely identifies the `div` element wrapping each product listing on the Poshmark site.

3.  **Locating Link Elements:** For each `tile` (individual listing container) identified in the previous step, the function needs to find the specific HTML element that contains the actual link to the product's detail page. This is commonly an `<a>` (anchor) tag. The method for finding this `<a>` tag within each `tile` would depend on its specific structure (e.g., `tile.find('a', class_='...')` or simply `tile.find('a')` if it's the only one).

4.  **Extracting `href` Attribute:** Once the relevant `<a>` tag is located within a `tile`, the function would extract its `href` attribute. This attribute holds the URL or relative path to the individual product page, for example, `link_element['href']`.

5.  **Ensuring Absolute URLs with `urljoin`:** It's crucial that all extracted URLs are absolute (e.g., `https://poshmark.com/listing/bag-name-123`). Often, the `href` attribute might contain a relative path (e.g., `/listing/bag-name-123`). To convert these relative paths into full, absolute URLs, the `urljoin` function from `urllib.parse` would be used. The `urljoin` function requires the base URL of the page being scraped (e.g., `https://poshmark.com`) and the relative path. It intelligently combines them to form a complete URL. This prevents issues if the scraper navigates to a new page and tries to use a relative link from a different context.

6.  **Return Value:** Finally, the function would collect all these absolute URLs into a list. To ensure efficiency and prevent redundant processing, it would typically convert this list to a set and then back to a list to remove any duplicate URLs, before returning the final list of unique, absolute product listing URLs.

### Subtask: Explain _extract_listing_links function

#### Instructions

The `_extract_listing_links` helper function is designed to systematically parse a Poshmark category page and return a list of unique, absolute URLs for each individual product listing found on that page.

1.  **Input:** The function would take a `BeautifulSoup` object, typically named `soup`, as its primary input. This `soup` object represents the parsed HTML content of a Poshmark category page (e.g., "Women-Bags").

2.  **Identifying Listing Containers:** To find all individual product listings, the function would utilize the `soup.find_all('div', class_="card--small")` method. As seen in the earlier examples, `card--small` is the CSS class that uniquely identifies the `div` element wrapping each product listing on the Poshmark site.

3.  **Locating Link Elements:** For each `tile` (individual listing container) identified in the previous step, the function needs to find the specific HTML element that contains the actual link to the product's detail page. This is commonly an `<a>` (anchor) tag. The method for finding this `<a>` tag within each `tile` would depend on its specific structure (e.g., `tile.find('a', class_='...')` or simply `tile.find('a')` if it's the only one).

4.  **Extracting `href` Attribute:** Once the relevant `<a>` tag is located within a `tile`, the function would extract its `href` attribute. This attribute holds the URL or relative path to the individual product page, for example, `link_element['href']`.

5.  **Ensuring Absolute URLs with `urljoin`:** It's crucial that all extracted URLs are absolute (e.g., `https://poshmark.com/listing/bag-name-123`). Often, the `href` attribute might contain a relative path (e.g., `/listing/bag-name-123`). To convert these relative paths into full, absolute URLs, the `urljoin` function from `urllib.parse` would be used. The `urljoin` function requires the base URL of the page being scraped (e.g., `https://poshmark.com`) and the relative path. It intelligently combines them to form a complete URL. This prevents issues if the scraper navigates to a new page and tries to use a relative link from a different context.

6.  **Return Value:** Finally, the function would collect all these absolute URLs into a list. To ensure efficiency and prevent redundant processing, it would typically convert this list to a set and then back to a list to remove any duplicate URLs, before returning the final list of unique, absolute product listing URLs.

### Subtask: Explain _extract_listing_links function

#### Instructions

The `_extract_listing_links` helper function is designed to systematically parse a Poshmark category page and return a list of unique, absolute URLs for each individual product listing found on that page.

1.  **Input:** The function would take a `BeautifulSoup` object, typically named `soup`, as its primary input. This `soup` object represents the parsed HTML content of a Poshmark category page (e.g., "Women-Bags").

2.  **Identifying Listing Containers:** To find all individual product listings, the function would utilize the `soup.find_all('div', class_="card--small")` method. As seen in the earlier examples, `card--small` is the CSS class that uniquely identifies the `div` element wrapping each product listing on the Poshmark site.

3.  **Locating Link Elements:** For each `tile` (individual listing container) identified in the previous step, the function needs to find the specific HTML element that contains the actual link to the product's detail page. This is commonly an `<a>` (anchor) tag. The method for finding this `<a>` tag within each `tile` would depend on its specific structure (e.g., `tile.find('a', class_='...')` or simply `tile.find('a')` if it's the only one).

4.  **Extracting `href` Attribute:** Once the relevant `<a>` tag is located within a `tile`, the function would extract its `href` attribute. This attribute holds the URL or relative path to the individual product page, for example, `link_element['href']`.

5.  **Ensuring Absolute URLs with `urljoin`:** It's crucial that all extracted URLs are absolute (e.g., `https://poshmark.com/listing/bag-name-123`). Often, the `href` attribute might contain a relative path (e.g., `/listing/bag-name-123`). To convert these relative paths into full, absolute URLs, the `urljoin` function from `urllib.parse` would be used. The `urljoin` function requires the base URL of the page being scraped (e.g., `https://poshmark.com`) and the relative path. It intelligently combines them to form a complete URL. This prevents issues if the scraper navigates to a new page and tries to use a relative link from a different context.

6.  **Return Value:** Finally, the function would collect all these absolute URLs into a list. To ensure efficiency and prevent redundant processing, it would typically convert this list to a set and then back to a list to remove any duplicate URLs, before returning the final list of unique, absolute product listing URLs.

### Subtask: Explain _extract_listing_links function

#### Instructions

The `_extract_listing_links` helper function is designed to systematically parse a Poshmark category page and return a list of unique, absolute URLs for each individual product listing found on that page.

1.  **Input:** The function would take a `BeautifulSoup` object, typically named `soup`, as its primary input. This `soup` object represents the parsed HTML content of a Poshmark category page (e.g., "Women-Bags").

2.  **Identifying Listing Containers:** To find all individual product listings, the function would utilize the `soup.find_all('div', class_="card--small")` method. As seen in the earlier examples, `card--small` is the CSS class that uniquely identifies the `div` element wrapping each product listing on the Poshmark site.

3.  **Locating Link Elements:** For each `tile` (individual listing container) identified in the previous step, the function needs to find the specific HTML element that contains the actual link to the product's detail page. This is commonly an `<a>` (anchor) tag. The method for finding this `<a>` tag within each `tile` would depend on its specific structure (e.g., `tile.find('a', class_='...')` or simply `tile.find('a')` if it's the only one).

4.  **Extracting `href` Attribute:** Once the relevant `<a>` tag is located within a `tile`, the function would extract its `href` attribute. This attribute holds the URL or relative path to the individual product page, for example, `link_element['href']`.

5.  **Ensuring Absolute URLs with `urljoin`:** It's crucial that all extracted URLs are absolute (e.g., `https://poshmark.com/listing/bag-name-123`). Often, the `href` attribute might contain a relative path (e.g., `/listing/bag-name-123`). To convert these relative paths into full, absolute URLs, the `urljoin` function from `urllib.parse` would be used. The `urljoin` function requires the base URL of the page being scraped (e.g., `https://poshmark.com`) and the relative path. It intelligently combines them to form a complete URL. This prevents issues if the scraper navigates to a new page and tries to use a relative link from a different context.

6.  **Return Value:** Finally, the function would collect all these absolute URLs into a list. To ensure efficiency and prevent redundant processing, it would typically convert this list to a set and then back to a list to remove any duplicate URLs, before returning the final list of unique, absolute product listing URLs.


## Explain _parse_item_page function

### Subtask:
Elaborate on the `_parse_item_page` helper function, detailing its strategies for extracting the listing title, price, brand, and size from an individual item's HTML. This includes looking for specific HTML elements and falling back to meta tags or general text searches when necessary.


### Elaborating on the `_parse_item_page` helper function

To effectively scrape detailed information from individual product listings, a helper function like `_parse_item_page` would be crucial. This function would encapsulate the logic for extracting specific data points from the HTML of a single listing, making the main scraping loop cleaner and more modular. Here's a breakdown of its likely implementation:

1.  **Primary Input:** The `_parse_item_page` function would typically take one main argument: a `BeautifulSoup` object representing the parsed HTML of a single product listing. This object, often obtained by iterating through the results of a `find_all` call (like `listings` in the notebook), allows for easy navigation and searching within that specific listing's HTML structure.

2.  **Listing Title Extraction:** For the listing title, the function would replicate the successful pattern shown in the notebook. It would use `tile.find('div', class_="title__condition__container")` to locate the `div` element that holds the title. Once found, `.text.strip()` would be applied to extract the raw text content and remove any leading or trailing whitespace, ensuring a clean title string. An empty string (`''`) would be returned if the element is not found, handled by a `try-except` block.

3.  **Price Extraction:** Extracting the price involves a multi-step process, similar to the existing code. The function would first use `tile.find('div', class_="m--t--1")` to find the `div` element containing the price string. Then, a regular expression, such as `re.search(r'(?:[$])(\d{1,3}(?:,\d{3})?)', price_text)`, would be employed to accurately capture the numeric price value, discarding the currency symbol and any other extraneous text. The captured string would then have commas removed using `.replace(",","")` and be converted to a `float` using `float()`. A `try-except` block would catch any errors during this process (e.g., if the price element is missing or the regex fails to match), returning `np.nan` to indicate a missing or unparseable price.

4.  **Brand Extraction:** The brand information often requires more flexible strategies. The function would first attempt to find a dedicated HTML element (e.g., a `div`, `span`, or `a` tag) with a specific class or ID that consistently holds the brand name. For example, it might look for `<a class="brand-name">Brand Name</a>`. If a direct element isn't immediately obvious, the function could check for `<meta>` tags in the page's head, such as `<meta property="product:brand" content="Brand Name">`, or even parse structured data embedded in `<script type="application/ld+json">` tags, which often contain schema.org data including product brand. As a fallback, if none of these methods yield a result, the function would return an empty string or a placeholder like 'Unknown'. This process would be wrapped in a `try-except` block.

5.  **Size Category Extraction:** Similar to brand, size information might not always be in a consistently labeled element. The function would prioritize looking for specific HTML elements (e.g., a `div`, `span`, or `p` tag) that explicitly indicate the size, perhaps by having a class like `item-size` or being near a label like "Size:". It might also need to perform a broader text search within a general `div` containing product details, looking for keywords like "Small", "Medium", "Large", or specific dimensions. If the size information is not found through direct element access or pattern matching, the function would default to an empty string or 'N/A', again protected by a `try-except` block.

6.  **Robustness with `try-except` blocks:** Throughout the function, `try-except` blocks are critical for handling the unpredictable nature of web scraping. Each extraction attempt (for title, price, brand, size) would be enclosed within its own `try-except` block. This ensures that if a particular element is missing, malformed, or causes an error during processing, the function doesn't crash but instead assigns a default value (e.g., `''` for strings, `np.nan` for numbers) to that field, allowing the scraping process to continue for other items. This makes the scraper resilient to variations in website structure.

7.  **Function Output:** The `_parse_item_page` function would return a structured piece of data, most likely a dictionary where keys correspond to the extracted fields (e.g., `'listing'`, `'price'`, `'brand'`, `'size'`) and values are the extracted data. Alternatively, it could return a list, with the elements in a predefined order. This structured output for each item can then be easily collected by the main scraping function and consolidated into a larger DataFrame.

## Explain _parse_item_page function

### Subtask:
Elaborate on the `_parse_item_page` helper function, detailing its strategies for extracting the listing title, price, brand, and size from an individual item's HTML. This includes looking for specific HTML elements and falling back to meta tags or general text searches when necessary.


### Explaining the `_parse_item_page` Helper Function

To effectively scrape detailed information for each listing, a helper function like `_parse_item_page` would be invaluable. This function would take the `BeautifulSoup` object representing a *single item listing* (e.g., one of the `tile` objects from the `listings` list) as its primary input. Its purpose is to systematically extract various attributes like the listing title, price, brand, and size.

Here's a detailed breakdown of its strategies:

1.  **Primary Input:** The function's main input would be a `BeautifulSoup` object, typically derived from iterating through a list of individual item `div` tags found on a search results page (e.g., `tile` in `for tile in listings:`). This object represents the parsed HTML of a single product listing.

2.  **Listing Title Extraction:**
    *   **Strategy:** The listing title is typically found within a `div` tag with the class `title__condition__container`. The function would use `tile.find('div', class_="title__condition__container")` to locate this specific element.
    *   **Text Cleaning:** Once the element is found, the raw text content would be extracted using `.text` and then cleaned using `.strip()` to remove leading/trailing whitespace. For example: `element.text.strip()`.
    *   **Error Handling:** A `try-except` block would wrap this operation. If the element is not found (e.g., `find` returns `None`), an `AttributeError` might occur when trying to access `.text`. In such cases, the function would catch the error and assign an empty string `''` as the title, ensuring the program's robustness.

3.  **Price Extraction:**
    *   **Strategy:** The price is identified within a `div` tag that has the class `m--t--1`. The initial retrieval would be `tile.find('div', class_="m--t--1").text`.
    *   **Regular Expressions and Cleaning:** The extracted text often contains more than just the price. A regular expression, like `r'(?:[$])(\d{1,3}(?:,\d{3})?)'`, is used with `re.search` to find the numerical price value, explicitly looking for a dollar sign followed by digits and optional commas. The first capturing group `groups()[0]` would isolate the number. Any commas within the number would then be removed using `.replace(",", "")`.
    *   **Type Conversion:** The cleaned string price would be converted to a `float` using `float()` to allow for numerical analysis.
    *   **Error Handling:** This entire process is enclosed in a `try-except` block. If the element or price pattern is not found, or if conversion to float fails, `np.nan` (from `numpy`) would be assigned as the price, indicating a missing numerical value gracefully.

4.  **Brand Extraction:**
    *   **Strategy 1 (Direct Element):** The first attempt would be to look for a specific HTML element, perhaps a `span` or `a` tag, that explicitly contains the brand name and has a distinctive class or ID (e.g., `tile.find('a', class_='_brand-name_')`).
    *   **Strategy 2 (Meta Tags):** If a direct element isn't found, the function might inspect the HTML for `<meta>` tags, particularly those related to schema.org or Open Graph, which often include product details. For example, `soup.find('meta', property='product:brand')` might yield a tag whose `content` attribute holds the brand name.
    *   **Strategy 3 (JSON-LD):** Some modern websites embed structured data using JSON-LD within `<script type="application/ld+json">` tags. The function could parse this JSON to extract the brand from the `brand` field of a `Product` schema.
    *   **Fallback:** If none of these methods yield a brand, a default value like `''` (empty string) or `'Unknown'` would be assigned.
    *   **Error Handling:** Each strategy would be within `try-except` blocks to handle cases where elements or attributes are missing.

5.  **Size Category Extraction:**
    *   **Strategy 1 (Specific Element):** Similar to brand, the first approach is to look for a dedicated HTML element, such as a `span` or `div` tag, explicitly labeled as 'Size' with its value (e.g., `tile.find('div', class_='listing-size-label')`).
    *   **Strategy 2 (Pattern Matching):** If a specific element isn't clear, the function might search for patterns within broader text blocks of the listing. For instance, it could look for phrases like "Size: Small", "Size: Medium", etc., using regular expressions within `div` tags that contain general product details.
    *   **Fallback:** If no size information is found, a default value like `''` or `'Not Specified'` would be used.
    *   **Error Handling:** `try-except` blocks would manage potential `AttributeError` or `TypeError` if elements or patterns are not found.

6.  **Importance of `try-except` Blocks:** Throughout the function, `try-except` blocks are crucial for robustness. Web scraping is inherently fragile due to dynamic website structures. By anticipating potential errors (like `AttributeError` when `.find()` returns `None`, or `ValueError` during type conversion) and providing default values (`''` for strings, `np.nan` for numbers), the function prevents crashes and ensures that incomplete or malformed data points don't stop the entire scraping process. This makes the scraper resilient to minor changes in the website's HTML structure and ensures that every listing is processed, even if some information is missing.

7.  **Typical Output:** The `_parse_item_page` function would typically return a structured collection of the extracted data for a single listing. This could be a `list` (e.g., `[listing_name, price, brand, size]`) that can be directly appended to a larger list of rows, or a `dictionary` (e.g., `{'listing': listing_name, 'price': price, 'brand': brand, 'size': size}`) that can then be easily converted into a DataFrame row. The list output is often preferred when building a DataFrame row by row.

## Explain post-processing and export

### Subtask:
Describe the final steps in the script, which involve creating a pandas DataFrame from the collected data, ensuring correct column types, and exporting the DataFrame to a CSV file named `bags.csv`, along with printing a summary.


After successfully scraping data from multiple pages, the raw collected data, typically stored as a list of lists (where each inner list represents a row of extracted information), needs to be structured into a more usable format. The `pandas` library provides the `DataFrame` object, which is ideal for this purpose.

1.  **Creating a pandas DataFrame:**
    The list of lists `data` (or `newData` in the example code) can be easily converted into a pandas DataFrame using `pd.DataFrame(data, columns=['listing', 'price', 'brand', 'size'])`. It is crucial to explicitly specify the `columns` argument with appropriate names (e.g., 'listing', 'price', 'brand', 'size') to ensure the data is organized correctly and easily accessible.

2.  **Verifying and Converting Data Types:**
    Once the DataFrame is created, it's good practice to verify the data types of each column using `df.info()` or `df.dtypes`. For columns like 'price', it's essential to ensure they have a numerical data type, such as `float64`, to enable numerical operations (like calculating the mean). If prices were extracted as strings (e.g., because of currency symbols or commas), they must be converted to numeric types. The `np.nan` value from the `numpy` library is particularly useful for representing missing numerical data, ensuring the 'price' column can correctly handle cases where a price might not have been found.
    For text-based columns like 'listing', 'brand', and 'size', the `object` (string) dtype is appropriate. The `try-except` blocks used during data collection are vital here, as they allow `np.nan` or empty strings to be inserted when information is missing, preventing the scraper from crashing and maintaining consistent data types.

3.  **Exporting to CSV:**
    To save the processed data for future use, the DataFrame is exported to a CSV file. This is done using the `.to_csv()` method, for example, `bags.to_csv('bags.csv', index=False)`. The `index=False` argument is very important as it prevents pandas from writing the DataFrame's internal index as a column in the CSV file, which is often redundant and undesirable.

4.  **Printing a Summary:**
    Finally, printing a brief summary of the collected data provides immediate insights. This can include the total number of entries, which can be obtained using `len(df)`, and the mean price, calculated with `df['price'].mean()`. This step helps to quickly confirm that the scraping process was successful and that a reasonable amount of data was collected, and also gives a high-level overview of the data's characteristics.

## Final Task

### Subtask:
Provide a comprehensive explanation of the entire Python script, synthesizing the roles of each function and outlining the overall workflow for scraping Poshmark data.


## Summary:

### Q&A

*   **How does the `poshmark` function orchestrate the scraping process?**
    The `poshmark` function acts as the central orchestrator, taking a `start_url`, target number of listings (`n`), `sleep_between` requests, and `max_pages` as parameters. It uses `requests.Session()` for persistent connections, collects listing URLs from category pages, and iterates through them, using pagination logic (e.g., incrementing `max_id`) until `n` listings are reached or `max_pages` is hit. It extracts listing names, prices, brands, and sizes, returning a consolidated pandas DataFrame.

*   **How does the `_clean_price` helper function (or equivalent logic) extract and convert price values?**
    The price extraction logic uses a regular expression `r'(?:[$])(\d{1,3}(?:,\d{3})?)'` to identify and capture the numerical portion of a price string, handling dollar signs and thousands separators. It then removes commas (e.g., "1,234" becomes "1234") and converts the cleaned string to a `float`. `try-except` blocks ensure robustness, returning `np.nan` if the price is missing or malformed.

*   **What is the purpose of the `_get_soup` helper function (or equivalent logic)?**
    The `_get_soup` logic is responsible for making HTTP GET requests to a given URL with custom `headers` (like `User-Agent`) to mimic a web browser and avoid detection. It then parses the received HTML content into a `BeautifulSoup` object, which facilitates easy navigation and data extraction from the webpage's structure.

*   **How does the `_extract_listing_links` helper function identify and extract individual product listing URLs?**
    This function takes a `BeautifulSoup` object of a category page. It identifies individual listing containers using `div` tags with the class `"card--small"`. Within each container, it locates `<a>` tags, extracts their `href` attributes, and then uses `urllib.parse.urljoin` to convert any relative paths into unique, absolute URLs.

*   **What strategies does the `_parse_item_page` helper function use to extract detailed information (title, price, brand, size) from an individual item's HTML?**
    This function processes the `BeautifulSoup` object of a single listing.
    *   **Title**: Extracted from a `div` with class `"title__condition__container"`.
    *   **Price**: Extracted from a `div` with class `"m--t--1"`, refined using regex, cleaned, and converted to `float`.
    *   **Brand**: Primarily sought in dedicated HTML elements, with fallbacks to `<meta>` tags or JSON-LD data within `<script>` tags if not found directly.
    *   **Size**: Located in specific HTML elements or identified through pattern matching within general text blocks.
    All extractions are wrapped in `try-except` blocks to handle missing elements or errors, assigning default values like `''` for strings or `np.nan` for numerical data, ensuring robust scraping.

*   **What are the final steps for post-processing and exporting the collected data?**
    The collected data, initially a list of lists, is transformed into a pandas DataFrame, with explicit column names (`listing`, `price`, `brand`, `size`). Data types are verified, ensuring numerical columns like `price` are `float64` (using `np.nan` for missing values). Finally, the DataFrame is exported to a CSV file named `bags.csv` using `df.to_csv('bags.csv', index=False)`, and a summary (e.g., total entries and mean price) is printed.

### Data Analysis Key Findings

*   The web scraping solution leverages `requests` for HTTP requests, `BeautifulSoup` for HTML parsing, `re` for regular expression-based data extraction, and `pandas` for data structuring.
*   The `poshmark` function is the core orchestrator, managing pagination and recursive calls to scrape a target number of listings or pages.
*   **Price extraction** is sophisticated, using a specific regex (`r'(?:[$])(\d{1,3}(?:,\d{3})?)'`) to parse numerical values, handling dollar signs and commas, and converting them to `float` for analysis.
*   **Robust error handling** is consistently applied across all data extraction steps, using `try-except` blocks to assign default values (empty strings or `np.nan`) when elements are missing or malformed, preventing script crashes.
*   **URL management** for individual listings involves identifying listing containers (`div` with class `"card--small"`), extracting `href` attributes, and using `urllib.parse.urljoin` to ensure unique, absolute URLs for detailed scraping.
*   **Flexible data extraction strategies** for `brand` and `size` involve prioritizing specific HTML elements but also include fallbacks to `<meta>` tags, JSON-LD, or broader text pattern matching.
*   The final output is a structured pandas DataFrame, ensuring appropriate data types for each column, which is then saved to a CSV file (`bags.csv`) and summarized with key metrics like the total number of entries and the mean price.

### Insights or Next Steps

*   The comprehensive design with modular helper functions and robust error handling makes this scraper highly adaptable and resilient to minor website structure changes.
*   To further enhance the scraper, consider implementing more advanced pagination techniques (e.g., dynamic page loading, handling different `max_id` parameter names) and potentially integrating proxy rotation to avoid IP blocking for large-scale scraping.
